# Quick Demo: Deblurring with the Trained Models

This is a short demo that uses the models trained in the main experiment. Nothing gets trained here, it just loads the saved weights from Drive and deblurs a few images, so the whole thing runs in a few minutes.

Needed before running:
1. Runtime > Change runtime type > GPU
2. fpn_inception.h5 in your Google Drive root
3. The fine tuned models in Drive under checkpoints/deblurgan_pruning_full_v2 (the main notebook saves them there)
4. Optional: GOPRO_Large.zip at MyDrive/datasets/GOPRO_Large.zip for the sample images


In [ ]:
# get the model code and the packages
!git clone -q https://github.com/VITA-Group/DeblurGANv2.git
%cd DeblurGANv2
!pip -q install pretrainedmodels albumentations scikit-image

import torch
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# connect Google Drive and set the paths
from google.colab import drive
drive.mount('/content/drive')

WEIGHTS_PATH = '/content/drive/MyDrive/fpn_inception.h5'
CHECKPOINT_BASE = '/content/drive/MyDrive/checkpoints/deblurgan_pruning_full_v2'
GOPRO_ZIP = '/content/drive/MyDrive/datasets/GOPRO_Large.zip'


In [ ]:
# build the generator and load the baseline plus the three fine tuned models
import sys, os, glob
sys.path.insert(0, '.')

# skip the imagenet download, the checkpoint overwrites these weights anyway
# (guarded so re-running this cell does not wrap the wrapper)
import pretrainedmodels
if not getattr(pretrainedmodels.inceptionresnetv2, '_patched', False):
    _orig = pretrainedmodels.inceptionresnetv2
    def _no_download(num_classes=1001, pretrained='imagenet'):
        return _orig(num_classes=num_classes, pretrained=None)
    _no_download._patched = True
    pretrainedmodels.inceptionresnetv2 = _no_download

from models.networks import get_generator

def new_generator():
    return get_generator({'g_name': 'fpn_inception', 'norm_layer': 'instance',
                          'learn_residual': True, 'dropout': True, 'blocks': 9})

models = {}
baseline = new_generator()
ckpt = torch.load(WEIGHTS_PATH, map_location='cpu', weights_only=False)
baseline.load_state_dict(ckpt['model'])
models['baseline'] = baseline.cuda()
print('baseline loaded')

for tag in ['10pct', '30pct', '50pct']:
    path = glob.glob(os.path.join(CHECKPOINT_BASE, '**', f'*{tag}*finetuned*.pth'), recursive=True)[0]
    m = new_generator()
    m.load_state_dict(torch.load(path, map_location='cpu'))
    models[tag] = m.cuda()
    print(tag, 'loaded')


## Deblur a few test images

The images come straight out of the GoPro zip on Drive, no need to extract the whole dataset.


In [ ]:
# the deblur function, same preprocessing as the main notebook
import numpy as np, cv2
import torch.nn.functional as F

def deblur(model, img):
    # the official protocol runs the model in train mode
    x = torch.from_numpy((img.astype(np.float32) / 127.5 - 1.0).transpose(2, 0, 1)).unsqueeze(0).cuda()
    h, w = x.shape[2], x.shape[3]
    ph, pw = (32 - h % 32) % 32, (32 - w % 32) % 32
    if ph or pw:
        x = F.pad(x, (0, pw, 0, ph), mode='reflect')
    model.train()
    with torch.no_grad():
        out = model(x)[:, :, :h, :w]
    return ((out.squeeze().cpu().numpy().transpose(1, 2, 0) + 1) / 2 * 255).clip(0, 255).astype(np.uint8)

print('ready')


In [ ]:
# pick a few test images from the zip and run all four models on them
import zipfile
import matplotlib.pyplot as plt
from skimage.metrics import peak_signal_noise_ratio

zf = zipfile.ZipFile(GOPRO_ZIP)
blur_names = sorted(n for n in zf.namelist()
                    if n.endswith('.png') and 'test' in n.split('/') and 'blur' in n.split('/'))
assert blur_names, f'no test blur images found, first zip entries: {zf.namelist()[:3]}'
picks = blur_names[::max(1, len(blur_names) // 4)][:4]

def read_zip_image(name):
    data = np.frombuffer(zf.read(name), np.uint8)
    return cv2.cvtColor(cv2.imdecode(data, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)

titles = ['blurry input', 'sharp original', 'baseline', 'pruned 10%', 'pruned 30%', 'pruned 50%']
fig, axes = plt.subplots(len(picks), 6, figsize=(22, 3.2 * len(picks)))
for r, name in enumerate(picks):
    blur = read_zip_image(name)
    sharp = read_zip_image(name.replace('/blur/', '/sharp/'))
    row = [blur, sharp] + [deblur(models[t], blur) for t in ['baseline', '10pct', '30pct', '50pct']]
    for c, img in enumerate(row):
        ax = axes[r][c]
        ax.imshow(img)
        ax.axis('off')
        label = titles[c]
        if c >= 2:
            label += f' ({peak_signal_noise_ratio(sharp, img):.2f} dB)'
        ax.set_title(label, fontsize=9)
plt.tight_layout()
plt.show()


## Try your own photo

Upload any photo, blurry ones work best. It gets deblurred by the original model and by the 50% pruned one so you can compare them.


In [ ]:
# upload a photo and deblur it
from google.colab import files
uploaded = files.upload()

for fname in uploaded:
    img = cv2.cvtColor(cv2.imdecode(np.frombuffer(uploaded[fname], np.uint8), cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
    # shrink very large photos so the GPU does not run out of memory
    if max(img.shape[:2]) > 1600:
        scale = 1600 / max(img.shape[:2])
        img = cv2.resize(img, (int(img.shape[1] * scale), int(img.shape[0] * scale)))
    outputs = [img, deblur(models['baseline'], img), deblur(models['50pct'], img)]
    names = ['your photo', 'baseline output', 'pruned 50% output']
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    for ax, im, t in zip(axes, outputs, names):
        ax.imshow(im)
        ax.set_title(t, fontsize=12)
        ax.axis('off')
    plt.tight_layout()
    plt.show()


In [ ]:
# the final numbers from the thesis, measured on the full test set
import json
with open(os.path.join(CHECKPOINT_BASE, 'results_clean_full_train.json')) as f:
    res = json.load(f)
print(f"{'model':<12} {'PSNR (dB)':>10} {'SSIM':>8}")
for key, label in [('baseline', 'baseline'), ('10pct', 'pruned 10%'), ('30pct', 'pruned 30%'), ('50pct', 'pruned 50%')]:
    print(f"{label:<12} {res[key]['psnr']:>10.2f} {res[key]['ssim']:>8.3f}")


## Done

The pruned models still deblur even with a big part of their channels turned off, which is the point of the whole study. The quality difference you see here matches the table above.
